# Setup

In [ ]:
# Check GPUs
!nvidia-smi
# Check path
!pwd

In [ ]:
# Install libraries
%pip install -q faster-whisper "pyannote.audio==3.3.2" pandas soundfile
%pip install -q transformers

In [ ]:
# Unpack dataset
!gunzip -k ../data/agora/agora.gz 
!ls ../data/agora/ | head

In [ ]:
# Check file characteristics
import wave, glob

wavs = sorted(glob.glob("../00_data/agora/c003/*.wav", recursive=True))
print(len(wavs), "WAV files found")

AUDIO = wavs[0]                     
with wave.open(AUDIO) as w:
    print("File:       ", AUDIO)
    print("Sample rate:", w.getframerate(), "Hz")
    print("Channels:   ", w.getnchannels())
    print("Bit depth:  ", w.getsampwidth() * 8, "bit")
    print("Duration:   ", round(w.getnframes() / w.getframerate() / 60, 1), "min")

## HuggingFace Token

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face token: ")

# Transcribe

In [ ]:
import soundfile as sf

audio, sr = sf.read(AUDIO, dtype="float32")   # the sound as a list of numbers
assert sr == 16000 and audio.ndim == 1, "Expected 16 kHz mono"
print(f"Loaded {len(audio)/sr/60:.1f} min of audio")

In [ ]:
from faster_whisper import WhisperModel

asr = WhisperModel("large-v3-turbo", device="cuda", device_index=0,
                   compute_type="float16")        

segments, info = asr.transcribe(
    audio,
    language="ca",       
    vad_filter=True,     # ignore silence and music 
    condition_on_previous_text=False,   # avoids repetition loops
)

# `segments` only does the work when we loop over it
text_segments = [{"start": s.start, "end": s.end, "text": s.text.strip()}
                 for s in segments]
print(f"{len(text_segments)} text segments, {info.duration/60:.1f} min of audio")
print(text_segments[:3])

## Diarization

In [ ]:
import os
from dataclasses import dataclass
import huggingface_hub
import torch
import torchaudio

# Fix for PyTorch 2.6+ 
try:
  torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])
except AttributeError:
  pass

_orig_torch_load = torch.load


def _patched_torch_load(*args, **kwargs):
  kwargs["weights_only"] = False
  return _orig_torch_load(*args, **kwargs)


torch.load = _patched_torch_load

# Compatibility fixes for recent torchaudio versions 
if not hasattr(torchaudio, "AudioMetaData"):

  @dataclass
  class AudioMetaData:
    sample_rate: int
    num_frames: int
    num_channels: int
    bits_per_sample: int
    encoding: str

  torchaudio.AudioMetaData = AudioMetaData

if not hasattr(torchaudio, "list_audio_backends"):
  torchaudio.list_audio_backends = lambda: ["soundfile"]

if not hasattr(torchaudio, "set_audio_backend"):
  torchaudio.set_audio_backend = lambda *args, **kwargs: None

# Fix for huggingface_hub 'use_auth_token' removal
_orig_hf_hub_download = huggingface_hub.hf_hub_download


def _patched_hf_hub_download(*args, **kwargs):
  if "use_auth_token" in kwargs:
    kwargs["token"] = kwargs.pop("use_auth_token")
  return _orig_hf_hub_download(*args, **kwargs)


huggingface_hub.hf_hub_download = _patched_hf_hub_download

In [ ]:
import torch
from pyannote.audio import Pipeline

dia_pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=os.environ["HF_TOKEN"])
dia_pipe.to(torch.device("cuda:1"))

waveform = torch.from_numpy(audio).unsqueeze(0)
diar = dia_pipe({"waveform": waveform, "sample_rate": 16000})

turns = [(t.start, t.end, spk)
         for t, _, spk in diar.itertracks(yield_label=True)]
print(f"{len(turns)} speaker turns, speakers: {sorted({s for _, _, s in turns})}")

# Miauerge transcript + diarization results

In [ ]:
import pandas as pd

def best_speaker(seg):
    """Pick the speaker whose turn overlaps most with this text segment."""
    overlap = {}
    for start, end, spk in turns:
        o = min(end, seg["end"]) - max(start, seg["start"])
        if o > 0:
            overlap[spk] = overlap.get(spk, 0) + o
    return max(overlap, key=overlap.get) if overlap else "UNKNOWN"

df = pd.DataFrame(text_segments)
df["speaker_label"] = df.apply(best_speaker, axis=1)

df = df.rename(columns={"start": "start_time", "end": "end_time", "text": "transcript"})
df = df[["start_time", "end_time", "speaker_label", "transcript"]]

df.to_csv("transcript_speakers.csv", index=False)
df.head(10)

In [ ]:
# Check
print(df.speaker_label.value_counts())
print((df.speaker_label == "UNKNOWN").mean().round(3), "fraction without a speaker")

# Determine speaker gender

In [ ]:
# Arquitectura del modelo de género (ver src/pipeline/age_gender_model.py)
import sys, os
sys.path.append(os.path.abspath("../.."))

import numpy as np, torch
from transformers import Wav2Vec2Processor
from src.pipeline.age_gender_model import AgeGenderWav2Vec2

In [ ]:
# Load model in the GPU
name = "audeering/wav2vec2-large-robust-24-ft-age-gender"
DEVICE = "cuda:1"

proc = Wav2Vec2Processor.from_pretrained(name)
gmodel = AgeGenderWav2Vec2.from_pretrained(name).half().to(DEVICE).eval()   # .half() = float16

In [ ]:
CLASSES = ["female", "male", "child"]    

@torch.no_grad()
def gender_probs(start, end, max_chunk=15.0):
    """Probabilities [female, male, child] for audio between start and end (seconds).
    Long turns are cut into chunks of up to 15 s, to save GPU memory."""
    results, weights = [], []
    t = start
    while t < end:
        t2 = min(t + max_chunk, end)
        if t2 - t >= 0.5:                                  # skip tiny pieces
            chunk = audio[int(t*16000):int(t2*16000)]
            x = proc(chunk, sampling_rate=16000)["input_values"][0]
            x = torch.tensor(x, dtype=torch.float16, device=DEVICE)[None]
            _, p = gmodel(x)
            results.append(p[0].float().cpu().numpy())
            weights.append(t2 - t)
        t = t2
    if not results:
        return None, 0.0
    return np.average(results, axis=0, weights=weights), sum(weights)

In [ ]:
# Score every speaker
import pandas as pd

turns_df = pd.DataFrame(turns, columns=["start", "end", "speaker"])
turns_df["dur"] = turns_df.end - turns_df.start

rows = []
for spk, g in turns_df.groupby("speaker"):
    total_speech = g.dur.sum()
    probs, weights, used = [], [], 0.0
    for _, r in g[g.dur >= 1.0].sort_values("dur", ascending=False).iterrows():
        p, w = gender_probs(r.start, r.end)
        if p is not None:
            probs.append(p); weights.append(w); used += w
        if used >= 60:
            break
    if probs:
        p = np.average(probs, axis=0, weights=weights)
        rows.append({"speaker": spk, "speech_sec": round(total_speech, 1),
                     "analysed_sec": round(used, 1),
                     **{f"p_{c}": round(float(v), 3) for c, v in zip(CLASSES, p)}})
    else:
        rows.append({"speaker": spk, "speech_sec": round(total_speech, 1),
                     "analysed_sec": 0.0, "p_female": None, "p_male": None, "p_child": None})

spk_table = pd.DataFrame(rows).set_index("speaker")
spk_table

In [ ]:
# Turn probabilities into label
THRESHOLD = 0.8

def decide(r):
    if r.p_female is None or pd.isna(r.p_female):
        return "unknown"
    p = [r.p_female, r.p_male, r.p_child]
    return CLASSES[int(np.argmax(p))] if max(p) >= THRESHOLD else "unknown"

spk_table["gender"] = spk_table.apply(decide, axis=1)
print(spk_table[["speech_sec", "p_female", "p_male", "p_child", "gender"]])

In [ ]:
# Add label to the table
df["gender"] = df.speaker_label.map(spk_table.gender).fillna("unknown")
df.to_csv("transcript_speakers_gender.csv", index=False)
df.head(10)